In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sidpy as sid
import pyNSID as nsid
import h5py

In [ ]:
def axis_generator(start_letter='i', prefix='axis_'):
    """
    Generates strings with prefix + letters from start_letter to 'z'.
    
    Args:
        start_letter (str): starting letter (default 'i')
        prefix (str): string prefix (default 'axis_')
    
    Yields:
        str: e.g. "axis_i", "axis_j", ...
    """
    start_ord = ord(start_letter.lower())
    end_ord = ord('z')
    
    for c in range(start_ord, end_ord + 1):
        yield prefix + chr(c)

In [6]:
all_results = np.load('all_results.npy')

amp_mat = all_results[:,0,:,:]
phase_mat = all_results[:,1,:,:]
pr_mat = amp_mat*np.cos(phase_mat)

print(pr_mat.shape) #line_number, time step, x
#Make sidpy dataset from the amplitude, phase and PR mats

pr_mat_sid = sid.Dataset.from_array(pr_mat, name='pr_mat')
amp_mat_sid = sid.Dataset.from_array(amp_mat, name='amp_mat')
phase_mat_sid = sid.Dataset.from_array(phase_mat, name='phase_mat')

#make the dimension vectors
line_vector = 2.0E-6*np.linspace(-1,1,pr_mat.shape[0]) #position in y for the line
time_step_vector = np.arange(pr_mat.shape[1])
x_vector = np.linspace(0, 2.0E-6, pr_mat.shape[2])

for sid_dset in [pr_mat_sid, amp_mat_sid, phase_mat_sid]:
    sid_dset.data_type = 'spectral_image'  # supported
    sid_dset.units = 'a.u.'
    sid_dset.quantity = 'Piezoresponse'
    sid_dset.set_dimension(0, sid.Dimension(line_vector,
                                          name='y', units='um', quantity='Width',
                                          dimension_type='spatial'))
    sid_dset.set_dimension(1, sid.Dimension(time_step_vector,
                                          'time', units='Time Step', quantity='Time',
                                          dimension_type='spectral'))
    sid_dset.set_dimension(2, sid.Dimension(x_vector,
                                          'x', units='um', quantity = 'Length', dimension_type='spatial' ))

(50, 2, 80, 128)
(50, 80, 128)


In [ ]:
data = {'piezoresponse': pr_mat_sid}
h5_name = 'test_h5_file2.h5'

with h5py.File(h5_name, 'w') as h5_f:
    h5_group = h5_f.create_group('Measurement_Nexus')
    my_str = "NXsidpy"
    def_dset = h5_group.create_dataset('definition', data=my_str)
    
    for key in list(data.keys()):
        h5_g = h5_group.create_group(key)
        nsid.hdf_io.write_nsid_dataset(data[key], h5_g)
    

In [ ]:
h5_g = h5_f['Measurement_Nexus/piezoresponse/generic']
dset_dims = []
[dset_dims.append(dim) for dim in h5_g if dim !='generic']
print(dset_dims)

# Example usage:
gen = axis_generator()
axis_ij_names = []
for ind,dset in enumerate(dset_dims):
    axis_name = next(gen)
    axis_ij_names.append(axis_name)
    h5_g.copy(dset, axis_name)

axes_refs = np.array(
            axis_ij_names,
            dtype='object'
        )
h5_g.attrs["NX_class"] = "NXdata"
h5_g.attrs["axes"] = axes_refs

for ind in range(len(axis_ij_names)):
    lab = axis_ij_names[ind]+"_indices"
    h5_g.attrs[lab]=ind

for ind,axis in enumerate(axis_ij_names):
    h5_g[axis].attrs["units"] = h5_g[dset_dims[ind]].attrs['units']
    h5_g[axis].attrs["long_name"] = h5_g[dset_dims[ind]].attrs['name']

h5_g.attrs["default"] = "generic"
h5_f.attrs["default"] = key
       
del h5_f['Measurement_Nexus'][key]['generic']['generic'].attrs["DIMENSION_LIST"] #delete dimension list attribute


#Reminder:
1. pyNSID shoudl only write to a Nexus compatible file format
2. When we read the file back, we shoudl read it back as a sidpy dataset format. We need to make sure to be able to read back files that are NSID-NExus compatible and the older (standard) NSID format.